# Предпроектный анализ «было»: доступность и обеспеченность детсадами

Расчёты по слоям `data/processed` (EPSG:4326, метрика — EPSG:32635). Пешеходный граф строится из **нашей** УДС
(`roads` + планируемая 41А-082) — без внешних запросов, скорость 5 км/ч. Цели верификации — числа
презентации контроля 03.11: 30-мин зоны от станции и порта, обеспеченность детсадами (до 7 мин / до 15 мин).

In [7]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from loguru import logger
logger.remove()

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from shapely.geometry import LineString, Point, box

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
RESULTS.mkdir(exist_ok=True)
CRS, MCRS = 4326, 32635
SPEED_M_MIN = 5000 / 60  # 5 км/ч

blocks = gpd.read_file(DATA / "blocks.geojson")
buildings = gpd.read_file(DATA / "building.geojson")
kg = gpd.read_file(DATA / "services" / "kindergarten.geojson")
bnd = gpd.read_file(DATA / "boundary.geojson")
roads = gpd.read_file(DATA / "roads.geojson")
planned = gpd.read_file(DATA / "scenario" / "roads.geojson")
print(f"кварталы: {len(blocks)}, здания: {len(buildings)}, детсады: {len(kg)}, население: {int(buildings['population'].sum())}")

кварталы: 526, здания: 9151, детсады: 4, население: 13659


In [8]:
# --- пешеходный граф УДС: узлы = вершины (сшивка 40 м), рёбра = отрезки, weight = время, мин
# unary_union заранее нодит линии в точках пересечений — иначе улицы без общих вершин остаются несвязными
import shapely
minx, miny, maxx, maxy = bnd.total_bounds
r = pd.concat([roads.cx[minx:maxx, miny:maxy], planned], ignore_index=True)
noded = shapely.ops.unary_union(list(r.geometry.explode(index_parts=False).geometry))
lines = list(noded.geoms) if noded.geom_type == "MultiLineString" else [noded]

coords, remap, edges = [], {}, []
for line in lines:
    ids = []
    for x, y in line.coords:
        key = (round(x * 111320), round(y * 111320))  # сетка ~1 м
        if key not in remap:
            remap[key] = len(coords); coords.append((x, y))
        ids.append(remap[key])
    edges += list(zip(ids, ids[1:]))

tree = cKDTree(np.array(coords))
parent = list(range(len(coords)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
for i, j in tree.query_pairs(0.0004):  # ~40 м в градусах
    ri, rj = find(i), find(j)
    if ri != rj: parent[max(ri, rj)] = min(ri, rj)

pos_wgs = {}
for i, c in enumerate(coords):
    pos_wgs.setdefault(find(i), c)
pts_m = gpd.GeoSeries([Point(c) for c in pos_wgs.values()], crs=CRS).to_crs(MCRS)
pos_m = dict(zip(pos_wgs.keys(), pts_m))

G = nx.Graph()
for i, j in edges:
    a, b = find(i), find(j)
    if a == b: continue
    t = pos_m[a].distance(pos_m[b]) / SPEED_M_MIN
    if not G.has_edge(a, b) or G[a][b]["weight"] > t:
        G.add_edge(a, b, weight=t)
comps = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
print(f"граф: {G.number_of_nodes()} узлов, {G.number_of_edges()} рёбер, {len(comps)} компонент, крупнейшие: {comps[:5]}")

граф: 4366 узлов, 4453 рёбер, 82 компонент, крупнейшие: [3591, 131, 68, 52, 39]


In [9]:
# --- 30-минутные пешеходные зоны от станции и порта (полигон = достижимые отрезки УДС + 25 м)
station = Point(28.6309, 60.3634)
port = Point(28.7355, 60.3500)
pt_m = gpd.GeoSeries([station, port], crs=CRS).to_crs(MCRS)
node_pts = np.array([[p.x, p.y] for p in pos_m.values()])
ntree = cKDTree(node_pts)
names = list(pos_m)

def zone_polymin(src_m, minutes=30):
    _, k = ntree.query([src_m.x, src_m.y])
    src = names[int(k)]
    dist = nx.single_source_dijkstra_path_length(G, src, cutoff=minutes, weight="weight")
    segs = [LineString([pos_m[u], pos_m[v]]) for u, v, _ in G.edges(data=True)
            if u in dist and v in dist and max(dist[u], dist[v]) <= minutes]
    return gpd.GeoSeries(segs, crs=MCRS).buffer(25).union_all().convex_hull if segs else None

zones = gpd.GeoDataFrame(
    {"name": ["станция 30 мин", "порт 30 мин"],
     "geometry": [zone_polymin(pt_m.iloc[0]), zone_polymin(pt_m.iloc[1])]},
    crs=MCRS).to_crs(CRS)
zones.to_file(RESULTS / "isochrones_walk30.geojson", driver="GeoJSON")
print("площади, км²:", [round(a, 1) for a in (zones.to_crs(MCRS).area / 1e6)])

площади, км²: [3.3, 6.0]


In [10]:
# --- население в 30-мин зонах (верификация презентации: станция 5 047 жит., порт 0)
cent = buildings.to_crs(MCRS).geometry.centroid
pop_pts = gpd.GeoDataFrame({"population": buildings["population"].values}, geometry=cent, crs=MCRS).to_crs(CRS)
bz = gpd.sjoin(pop_pts, zones[["name", "geometry"]], predicate="within")
print(bz.groupby("name")["population"].sum())
print("проверка презентации: станция 5 047, порт 0")

name
порт 30 мин        444
станция 30 мин    3468
Name: population, dtype: int32
проверка презентации: станция 5 047, порт 0


In [11]:
# --- население и вместимость детсадов по кварталам
blk = gpd.GeoDataFrame(geometry=blocks.to_crs(MCRS).geometry.centroid, crs=MCRS).to_crs(CRS)
bpt = gpd.GeoDataFrame({"population": buildings["population"].values}, geometry=cent, crs=MCRS).to_crs(CRS)
bj = gpd.sjoin(blocks[["geometry"]], bpt, predicate="contains")
pop_blk = bj.groupby(level=0)["population"].sum().reindex(blocks.index, fill_value=0)
print(f"население по кварталам: сумма {int(pop_blk.sum())} из {int(buildings['population'].sum())} "
      f"(остаток — вне кварталов)")
print("детсады:", kg[["service_name", "capacity"]].to_string(index=False) if len(kg) else "—")
print("вместимость детсадов всего:", int(kg["capacity"].sum()))

население по кварталам: сумма 13597 из 13659 (остаток — вне кварталов)
детсады: service_name  capacity
 Детский сад       160
 Детский сад       110
 Детский сад       100
 Детский сад       100
вместимость детсадов всего: 470


In [12]:
# --- матрица доступности между кварталами, мин пешком (центроид → ближайший узел графа)
blk_m = blocks.to_crs(MCRS).geometry.centroid
_, kn = ntree.query(np.array([[p.x, p.y] for p in blk_m]))
src_nodes = [names[int(k)] for k in kn]
acc = np.full((len(blocks), len(blocks)), np.inf, dtype=np.float32)
approach = []
for i, s in enumerate(src_nodes):
    d = nx.single_source_dijkstra_path_length(G, s, weight="weight")
    row = np.array([d.get(t, np.inf) for t in src_nodes], dtype=np.float32)
    approach.append(abs(pos_m[s].distance(blk_m.iloc[i])) / SPEED_M_MIN)
    acc[i] = row + approach[-1]
acc_mx = pd.DataFrame(acc, index=blocks.index, columns=blocks.index)
print("матрица", acc_mx.shape, "| медиана времени:", round(float(np.median(acc[acc < np.inf])), 1), "мин")

матрица (526, 526) | медиана времени: 343.9 мин


In [13]:
# --- обеспеченность детсадами: BlocksNet competitive_provision
from blocksnet.analysis.provision import competitive_provision
print(competitive_provision.__doc__[:1200] if competitive_provision.__doc__ else "нет docstring")

нет docstring


In [15]:
# --- обеспеченность детсадами: BlocksNet competitive_provision
# нормативы Ленобласти: 61 место/1000 (город), 40 (посёлки); границы города в слоях нет — считаем диапазон.
from blocksnet.analysis.provision import competitive_provision
import inspect
print("сигнатура:", inspect.signature(competitive_provision))

SCEN_KG = gpd.read_file(DATA / "scenario" / "kindergarten.geojson")

def cap_by_block(kgs):
    cap = pd.concat([k[["capacity", "geometry"]] for k in kgs], ignore_index=True)
    cb = gpd.sjoin(blocks[["geometry"]], gpd.GeoDataFrame(cap, geometry="geometry", crs=CRS),
                   predicate="contains")
    return cb.groupby(level=0)["capacity"].sum().reindex(blocks.index, fill_value=0)

def provision_summary(kgs, norm, accessibility):
    cap_blk = cap_by_block(kgs)
    df = pd.DataFrame({"population": pop_blk, "capacity": cap_blk})
    prov, _ = competitive_provision(df, acc_mx, accessibility=accessibility, demand=norm)
    need = float(prov["demand"].sum())
    left = float(prov["demand_left"].sum())
    return {"мест": int(cap_blk.sum()), "потребность": round(need),
            "покрыто": round(need - left), "доля": round(1 - left / need, 2),
            "дефицит": round(left)}

baseline = [kg]
scen = [kg, SCEN_KG]
print("=== ДО (470 мест) ===")
for norm in (61, 40):
    print(norm, "/1000, пешком 7 мин:", provision_summary(baseline, norm, 7))
    print(norm, "/1000, подвоз 15 мин:", provision_summary(baseline, norm, 15))
print("=== ПОСЛЕ (+детсад 160 мест, Карасёвка) ===")
for norm in (61, 40):
    print(norm, "/1000, пешком 7 мин:", provision_summary(scen, norm, 7))
    print(norm, "/1000, подвоз 15 мин:", provision_summary(scen, norm, 15))
print("проверка презентации: 470 мест, потребность 681 (смешанная норма), 29% пешком / 62% с подвозом,")
print("после: 262→107 дефицит, 62→84% с подвозом")

сигнатура: (blocks_df: pandas.core.frame.DataFrame, accessibility_matrix: pandas.core.frame.DataFrame, accessibility: int, demand: int | None = None, self_supply: bool = True, max_depth: int = 1) -> tuple[pandas.core.frame.DataFrame, pandas.core.frame.DataFrame]
=== ДО (470 мест) ===
61 /1000, пешком 7 мин: {'мест': 470, 'потребность': 937, 'покрыто': 163, 'доля': 0.17, 'дефицит': 774}
61 /1000, подвоз 15 мин: {'мест': 470, 'потребность': 937, 'покрыто': 232, 'доля': 0.25, 'дефицит': 705}
40 /1000, пешком 7 мин: {'мест': 470, 'потребность': 664, 'покрыто': 110, 'доля': 0.17, 'дефицит': 554}
40 /1000, подвоз 15 мин: {'мест': 470, 'потребность': 664, 'покрыто': 157, 'доля': 0.24, 'дефицит': 507}
=== ПОСЛЕ (+детсад 160 мест, Карасёвка) ===
61 /1000, пешком 7 мин: {'мест': 630, 'потребность': 937, 'покрыто': 185, 'доля': 0.2, 'дефицит': 752}
61 /1000, подвоз 15 мин: {'мест': 630, 'потребность': 937, 'покрыто': 269, 'доля': 0.29, 'дефицит': 668}
40 /1000, пешком 7 мин: {'мест': 630, 'потреб

In [16]:
# --- диагностика семантики demand в competitive_provision
import inspect
src = inspect.getsource(competitive_provision)
for ln in src.splitlines():
    if any(k in ln.lower() for k in ("demand", "1000", "/ ", "capacity")):
        print(ln)

    demand: int | None = None,
    blocks_df = _initialize_provision_df(blocks_df, demand)
        blocks_df, depth_links = _distribute_demand(blocks_df, accessibility_matrix, accessibility, depth)
        break_condition = blocks_df[DEMAND_LEFT_COLUMN].sum() == 0 or blocks_df[CAPACITY_LEFT_COLUMN].sum() == 0
    blocks_df[PROVISION_STRONG_COLUMN] = blocks_df[DEMAND_WITHIN_COLUMN] / blocks_df.demand
        blocks_df[DEMAND_WITHIN_COLUMN] + blocks_df[DEMAND_WITHOUT_COLUMN]
    ) / blocks_df.demand


In [17]:
# --- подвоз — на транспорте: та же сеть, скорость автобуса 25 км/ч (время пешком × 0,2)
acc_bus = acc_mx * (5 / 25)

def prov_run(kgs, norm, acc, accessibility):
    cap_blk = cap_by_block(kgs)
    df = pd.DataFrame({"population": pop_blk, "capacity": cap_blk})
    prov, _ = competitive_provision(df, acc, accessibility=accessibility, demand=norm)
    need = float(prov["demand"].sum()); left = float(prov["demand_left"].sum())
    return round(1 - left / need, 2), round(left)

print("=== ДО: пешком 7 мин / подвоз 15 мин на автобусе ===")
for norm in (61, 40):
    w = prov_run([kg], norm, acc_mx, 7)
    b = prov_run([kg], norm, acc_bus, 15)
    need = round(pop_blk.sum() * norm / 1000)
    print(f"норма {norm}/1000: потребность ≈{need}, мест 470 | пешком: {w[0]:.0%}, дефицит {w[1]} | "
          f"подвоз: {b[0]:.0%}, дефицит {b[1]}")
print("=== ПОСЛЕ (+детсад 160, Карасёвка) ===")
for norm in (61, 40):
    w = prov_run([kg, SCEN_KG], norm, acc_mx, 7)
    b = prov_run([kg, SCEN_KG], norm, acc_bus, 15)
    print(f"норма {norm}/1000: мест 630 | пешком: {w[0]:.0%}, дефицит {w[1]} | "
          f"подвоз: {b[0]:.0%}, дефицит {b[1]}")
print("презентация (для сверки): до — 29% пешком / 62% подвоз, дефицит 262; после — 84% подвоз, дефицит 107")

=== ДО: пешком 7 мин / подвоз 15 мин на автобусе ===
норма 61/1000: потребность ≈829, мест 470 | пешком: 17%, дефицит 774 | подвоз: 40%, дефицит 559
норма 40/1000: потребность ≈544, мест 470 | пешком: 17%, дефицит 554 | подвоз: 46%, дефицит 356
=== ПОСЛЕ (+детсад 160, Карасёвка) ===
норма 61/1000: мест 630 | пешком: 20%, дефицит 752 | подвоз: 49%, дефицит 480
норма 40/1000: мест 630 | пешком: 19%, дефицит 539 | подвоз: 48%, дефицит 348
презентация (для сверки): до — 29% пешком / 62% подвоз, дефицит 262; после — 84% подвоз, дефицит 107


In [18]:
# --- сохранение сводки и данных для следующих тетрадей/карт
rows = []
for label, kgs in (("до", [kg]), ("после", [kg, SCEN_KG])):
    for norm in (61, 40):
        w = prov_run(kgs, norm, acc_mx, 7)
        b = prov_run(kgs, norm, acc_bus, 15)
        rows.append({"сценарий": label, "норма_мест_на_1000": norm,
                     "мест": 470 if label == "до" else 630,
                     "потребность": round(pop_blk.sum() * norm / 1000),
                     "доля_пешком_7мин": w[0], "дефицит_пешком": w[1],
                     "доля_подвоз_15мин": b[0], "дефицит_подвоз": b[1]})
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / "provision_kindergarten.csv", index=False, encoding="utf-8-sig")
acc_mx.to_pickle(RESULTS / "acc_mx_walk.pkl")
acc_bus.to_pickle(RESULTS / "acc_mx_bus.pkl")
pop_blk.to_pickle(RESULTS / "pop_by_block.pkl")
summary

,сценарий,норма_мест_на_1000,мест,потребность,доля_пешком_7мин,дефицит_пешком,доля_подвоз_15мин,дефицит_подвоз
0,до,61,470,829,0.17,774,0.40,559
1,до,40,470,544,0.17,554,0.46,356
2,после,61,630,829,0.20,752,0.49,480
3,после,40,630,544,0.19,539,0.48,348
